# Species-Area Relationship Analysis: Southern Appalachian Amphibians

Analysis of amphibian species richness as a function of area across 26 localities in the Southern Appalachian region, using the `sars` library for model fitting and comparison.

In [ ]:
import sars
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

## 1. Load and Explore the Data

In [ ]:
raw = pd.read_csv("data/species_area_2025.csv")
raw.describe()

## 2. Prepare Data for `sars`

The library expects columns named `area` and `species`. We'll analyse total amphibians first, then frogs and salamanders separately.

In [ ]:
data_amphibians = sars.from_csv("data/species_area_2025.csv", area_col="Area", species_col="AmphibianSpecies")
data_frogs = sars.from_csv("data/species_area_2025.csv", area_col="Area", species_col="FrogSpecies")
data_salamanders = sars.from_csv("data/species_area_2025.csv", area_col="Area", species_col="SalamanderSpecies")

data_amphibians.head()

## 3. Classic Power-Law SAR (S = cA^z)

In [ ]:
power_fit = sars.sar_power(data_amphibians)
print(power_fit)
print(f"\nParameters: {power_fit.params}")
print(f"R²: {power_fit.r_squared:.4f}")
print(f"AICc: {power_fit.aicc:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sars.plot_fit(power_fit, log=False, ax=axes[0])
axes[0].set_title("Power Model (arithmetic)")
axes[0].set_xlabel("Area (km²)")
axes[0].set_ylabel("Amphibian Species")

sars.plot_fit(power_fit, log=True, ax=axes[1])
axes[1].set_title("Power Model (log-log)")
axes[1].set_xlabel("log Area")
axes[1].set_ylabel("log Species")

plt.tight_layout()
plt.show()

## 4. Multi-Model Comparison

Fit all 20 candidate SAR models and rank by AICc.

In [ ]:
multi = sars.sar_multi(data_amphibians)
print(multi)
multi.summary

In [ ]:
sars.plot_multi(multi, top_n=5)
plt.title("Top 5 SAR Models by AICc")
plt.xlabel("Area (km²)")
plt.ylabel("Amphibian Species")
plt.show()

## 5. Model Averaging with Bootstrap Confidence Intervals

In [ ]:
avg = sars.sar_average(data_amphibians)

print("Akaike weights (top 5):")
for model, w in sorted(avg.weights.items(), key=lambda x: -x[1])[:5]:
    print(f"  {model:12s}  {w:.4f}")

In [ ]:
boot = sars.bootstrap_ci(data_amphibians, n_boot=1000, conf=0.95, rng=np.random.default_rng(42))

sars.plot_average(avg, ci=True, boot=boot)
plt.title("Model-Averaged SAR with 95% Bootstrap CI")
plt.xlabel("Area (km²)")
plt.ylabel("Amphibian Species")
plt.show()

## 6. Small Island Effect (Threshold / Piecewise Models)

In [ ]:
thresh = sars.sar_threshold(data_amphibians)
print(thresh)
print(f"\nBest model: {thresh.best_model}")
print(f"Breakpoint (log-area): {thresh.best_breakpoint:.3f}")
print(f"Segments: {thresh.best_segments}")
thresh.summary

## 7. Residual Diagnostics

In [ ]:
sars.plot_residuals(power_fit)
plt.title("Residuals — Power Model")
plt.show()

## 8. Frogs vs. Salamanders

Compare SAR patterns between the two amphibian orders.

In [ ]:
frog_power = sars.sar_power(data_frogs)
sal_power = sars.sar_power(data_salamanders)

print("Frogs:       ", frog_power.params, f"  R²={frog_power.r_squared:.4f}")
print("Salamanders: ", sal_power.params, f"  R²={sal_power.r_squared:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sars.plot_fit(frog_power, log=True, ax=axes[0])
axes[0].set_title("Frogs — Power Model (log-log)")
axes[0].set_xlabel("log Area")
axes[0].set_ylabel("log Species")

sars.plot_fit(sal_power, log=True, ax=axes[1])
axes[1].set_title("Salamanders — Power Model (log-log)")
axes[1].set_xlabel("log Area")
axes[1].set_ylabel("log Species")

plt.tight_layout()
plt.show()

## 9. Multi-Model Comparison — Frogs & Salamanders

In [ ]:
multi_frogs = sars.sar_multi(data_frogs)
multi_sal = sars.sar_multi(data_salamanders)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sars.plot_multi(multi_frogs, top_n=5, ax=axes[0])
axes[0].set_title("Frogs — Top 5 Models")
axes[0].set_xlabel("Area (km²)")
axes[0].set_ylabel("Frog Species")

sars.plot_multi(multi_sal, top_n=5, ax=axes[1])
axes[1].set_title("Salamanders — Top 5 Models")
axes[1].set_xlabel("Area (km²)")
axes[1].set_ylabel("Salamander Species")

plt.tight_layout()
plt.show()

In [ ]:
print("=== Frogs — Top 5 ===")
display(multi_frogs.summary.head())
print("\n=== Salamanders — Top 5 ===")
display(multi_sal.summary.head())